In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.auto import tqdm
from itertools import product

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '4_Baselines' / '4.2_OutcomeModel'))
import MF_class as MF
import OM_class as OM

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### Score function

$$
s(u,i,j,x) = [p_u; q_i]^{\top} (W + x\Delta) q_j + \alpha_u b_u + \alpha_i b_i + \alpha_j b_j + \alpha_g 
$$

where:

- $u$ : user index  
- $i$ : treatment item  
- $j$ : outcome item
- $x \in \{0,1\}$ : indicator for $u$ interacting with $i$ **and** not intercating with $j$ *before* $i$

- $p_u, q_i, q_j  \in \mathbb{R}^d$ : embedding vectors of user $u$, item $i$, item $j$

- $[p_u; q_i] \in \mathbb{R}^{2d}$ : concatenation of the user and treatment item embeddings

- $W \in \mathbb{R}^{2d \times d}$ : baseline pathway matrix mapping $(u,i)$ to item $j$  
- $\Delta \in \mathbb{R}^{2d \times d}$ : incremental pathway effect under treatment ($x=1$)

- $b_u, b_i, b_j \in \mathbb{R}$ : user, treatment item, and outcome item bias terms

- $\alpha_u, \alpha_i, \alpha_j \in \mathbb{R}$ : scaling coefficients for the bias terms

- $\alpha_g \in \mathbb{R}$ : global bias

- $s(u,i,j,x)$ : predicted logit score for the interaction.

---

### Training loss

$$
\mathcal{L}
= - \sum_n \left[
y_n \log \sigma(s_n)
+ (1 - y_n)\log(1 - \sigma(s_n))
\right]
$$

where:

- $n$ : index of a training observation  
- $y_n \in \{0,1\}$ : observed label (interaction with $j$, occurred or not)  
- $s_n$ : model logit score for observation $n$  
- $\sigma(z) = \frac{1}{1 + e^{-z}}$ : logistic sigmoid function  
- $\mathcal{L}$ : binary cross-entropy loss.

# 1. Load Dataset

In [2]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Sequels'

In [3]:
om_train_data = pd.read_csv(data_path / 'om_train.csv')
om_test_data  = pd.read_csv(data_path / 'om_test.csv')

# 2. Load MF Embeddings

In [5]:
MF_model = MF.MatrixFactorizationTorch(
    n_users=7801,
    n_items=6384,
    n_factors=25,
)

model_path = base_artifacts / 'Propensity_Models'
model_name = f'MF_sequels'
MF_model.load(path=model_path / (model_name + '.pt'))

user_embeddings = MF_model.P.detach().numpy()[:-1]
item_embeddings = MF_model.Q.detach().numpy()[:-1]
user_bias = MF_model.b_u.detach().numpy()[:-1]
item_bias = MF_model.b_i.detach().numpy()[:-1]

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          25
Learning rate:              0.0005
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           40
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 14:17:00


# 3. Train Outcome Model

In [6]:
model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

model.fit(
    df_train=om_train_data,
    df_valid=om_test_data,
    lr=2e-4,
    weight_decay=1e-4,
    epochs=20,
    batch_size=2**13,
)

/home/gouni/CausalI2I/4_Baselines/4.3_OutcomeModel/OM_class.py:247: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and is_cuda))


Loss: BCE
Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || Loss   | L-POS   | L-NEG   | MPR    || Loss   | L-POS   | L-NEG   | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
    1  || 0.5426 |  0.6751 |  0.4101 | 0.8179 || 0.5141 |  0.5614 |  0.4191 | 0.8568 ||  0.9558 | None  |     7.7s
    2  || 0.4335 |  0.4603 |  0.4067 | 0.8911 || 0.4606 |  0.4974 |  0.3866 | 0.8812 ||  0.5526 | 0.833 |    15.1s
    3  || 0.3944 |  0.4077 |  0.3810 | 0.9087 || 0.4373 |  0.4732 |  0.3654 | 0.8910 ||  0.4139 | 0.956 |    22.5s
    4  || 0.3720 |  0.3796 |  0.3644 | 0.9173 || 0.4207 |  0.4516 |  0.3588 | 0.8964 ||  0.3386 | 0.967 |    29.9s
    5  || 0.3570 |  0.3605 |  0.3535 | 0.9227 || 0.4135 |  0.4473 |  0.3457 | 0.8999 ||  0.2967 | 0.974 |    37.3s
    6  || 0.3461 |  0.3474 |  0.3447 | 0.9265 || 0.4088 |  0.4454 |

In [10]:
model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

model.fit(
    df_train=om_train_data,
    df_valid=om_test_data,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=40,
    batch_size=2**13,
)

Loss: BCE
Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || Loss   | L-POS   | L-NEG   | MPR    || Loss   | L-POS   | L-NEG   | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
    1  || 0.5934 |  0.7945 |  0.3924 | 0.7802 || 0.5866 |  0.6671 |  0.4252 | 0.8202 ||  0.6022 | None  |     7.4s
    2  || 0.4895 |  0.5526 |  0.4264 | 0.8606 || 0.5145 |  0.5638 |  0.4155 | 0.8573 ||  0.3923 | 0.874 |    14.7s
    3  || 0.4455 |  0.4779 |  0.4131 | 0.8856 || 0.4804 |  0.5212 |  0.3985 | 0.8728 ||  0.3090 | 0.968 |    22.1s
    4  || 0.4184 |  0.4416 |  0.3952 | 0.8984 || 0.4577 |  0.4916 |  0.3896 | 0.8816 ||  0.2596 | 0.983 |    29.5s
    5  || 0.3996 |  0.4134 |  0.3857 | 0.9065 || 0.4443 |  0.4779 |  0.3769 | 0.8874 ||  0.2264 | 0.984 |    36.8s
    6  || 0.3854 |  0.3949 |  0.3759 | 0.9123 || 0.4354 |  0.4706 |

## Save Model

In [11]:
model_path = base_artifacts / 'Outcome_Models' / f'OM_sequels.pt'

In [12]:
model.save(model_path, note="none")

## Load Model

In [13]:
loaded_model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

loaded_model.load(model_path)

Loaded OutcomeModel summary:
Model:             OutcomeModel
Embedding dim:     25
Loss:              BCE
Learning rate:     0.0001
Weight decay:      0.0001
Batch size:        8192
Epochs:            40
Use AMP:           True
Timestamp:         2026-04-13 14:45:19
Note:              none
